In [ ]:
# MODELADO FINAL ECONOMÉTRICO PARA 18 bases EN PARAGUAY (2000-2023)
# Test de Pesaran (Dependencia Espacial) y Regresión de Driscoll-Kraay (HAC/CSD)

import pandas as pd
import numpy as np
import scipy.stats as stats
from linearmodels.panel import RandomEffects, PanelOLS

# 1. Cargar y preparar datos mensuales procesados
df = pd.read_csv('/home/hjvargaso/proyectos/tesis_sequia_2/data/processed/datos_mensuales_con_spei.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

# Diccionario definitivo con los clústeres reales optimizados de su ejecución (18 bases)
cluster_mapping = {
    'MINGA': 1, 'SJBA': 1, 'VILL': 1, 'CAAZ': 1, 'COVIE': 1, 'CMEZA': 1, 'ENCAR': 1,  # Clúster 1 (Sur-Este)
    'MCAL': 2, 'SEST': 2, 'SPED': 2, 'GBRU': 2, 'CONC': 2, 'PCOL': 2, 'LUQUE': 2, 'PJC': 2, 'PTOC': 2, 'PILAR': 2, 'SGUA': 2 # Clúster 2 (Norte-Centro-Chaco)
}

# Crear la columna 'cluster' directamente usando .map()
df['cluster'] = df['estacion_id'].map(cluster_mapping)

# Codificación de las Variables Explicativas (Regresores del Modelo 3.14)
# A. Tendencia temporal secuencial (tiempo_t: mes 1 a 288)
df['tiempo'] = (df['fecha'].dt.year - 2000) * 12 + df['fecha'].dt.month

# B. Dummies estacionales climáticas (estacion_climatica)
# El invierno actúa como categoría de referencia (omitida en la regresión)
df['estacion_climatica'] = np.where(df['fecha'].dt.month.isin([12, 1, 2]), 'verano',
                           np.where(df['fecha'].dt.month.isin([3, 4, 5]), 'otono',
                           np.where(df['fecha'].dt.month.isin([9, 10, 11]), 'primavera', 'invierno')))

# C. Dummies regionales geográficas basadas en los Clústeres (region-geografica)
# El Clúster 2 (Noroeste-Centro-Chaco) actúa como categoría de referencia
df['region_geografica'] = df['cluster'].astype(str)

# Configurar índice doble estructural para Datos de Panel
df_panel = df.set_index(['estacion_id', 'fecha'])

# 2. Construcción de Matrices de Diseño usando Patsy (Ecuación 3.14)
from patsy import dmatrices
formula = "spei_6 ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"
y, X = dmatrices(formula, df_panel, return_type='dataframe')


# 3. VERIFICACIÓN DE DEPENDENCIA ESPACIAL: TEST DE CD DE PESARAN (2004)

model_preliminar = RandomEffects(y, X).fit()

# CORRECCIÓN DE LA ASIGNACIÓN:
# Al asignar la Serie (con su índice) en lugar del arreglo de numpy (.values), 
# Pandas alinea automáticamente los índices y rellena con NaNs las filas descartadas
df_panel['residuos'] = model_preliminar.resids

# Restablecemos el índice para poder pivotar la matriz de residuos de forma segura
df_residuos = df_panel.reset_index()
residuos_pivot = df_residuos.pivot(index='fecha', columns='estacion_id', values='residuos').dropna()

T = residuos_pivot.shape[0]
N = residuos_pivot.shape[1]

# Matriz de correlaciones de Pearson
matriz_corr = residuos_pivot.corr(method='pearson')
upper_tri_indices = np.triu_indices(N, k=1)
suma_rho = matriz_corr.values[upper_tri_indices].sum()

# Estadístico CD de Pesaran
pesaran_cd = np.sqrt(2 * T / (N * (N - 1))) * suma_rho
p_val_cd = 2 * (1 - stats.norm.cdf(abs(pesaran_cd)))

print("\n========================================================")
print("TEST DE DEPENDENCIA DE SECCIÓN CRUZADA DE PESARAN (2004)")
print("========================================================")
print(f"Estadístico CD de Pesaran: {pesaran_cd:.4f}")
print(f"p-valor resultante:        {p_val_cd:.4e}")
print("--------------------------------------------------------")
if p_val_cd < 0.05:
    print("Decisión: Se rechaza H0 (Independencia de Sección Cruzada).")
    print("Existe una fuerte correlación espacial/CSD entre estaciones.")
    print("Metodología: Se requiere el estimador robusto de Driscoll-Kraay.")
else:
    print("Decisión: No se rechaza H0. Hay independencia espacial.")
print("========================================================\n")

# 4. ESTIMACIÓN ROBUSTA FINAL: ERRORES ESTÁNDAR DE DRISCOLL-KRAAY (1998)

# 'drop_absorbed=True' remueve de efectos fijos variables de nivel redundantes de forma segura
fe_model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True).fit()
re_model_dk = RandomEffects(y, X).fit(cov_type='kernel', kernel='bartlett')

print(re_model_dk)


# 5. EXPORTACIÓN DINÁMICA DE TABLAS DE LATEX CON LOS RESULTADOS REALES

coef = re_model_dk.params
std_err = re_model_dk.std_errors
pvalues = re_model_dk.pvalues
tstats = re_model_dk.tstats

# Diccionario de traducción para limpiar variables de Patsy
translation_map = {
    "Intercept": "Intercepto ($\\beta_0$)",
    "C(estacion_climatica, Treatment('invierno'))[T.otono]": "Oto{\\~n}o ($\\gamma_1$)",
    "C(estacion_climatica, Treatment('invierno'))[T.primavera]": "Primavera ($\\gamma_2$)",
    "C(estacion_climatica, Treatment('invierno'))[T.verano]": "Verano ($\\gamma_3$)",
    "C(region_geografica, Treatment('2'))[T.1]": "Regi{\\'o}n Oriental / Cl{\\'u}ster 1 ($\\delta_1$)",
    "C(estacion_climatica, Treatment('invierno'))[T.otono]:C(region_geografica, Treatment('2'))[T.1]": "Oto{\\~n}o $\\times$ Cl{\\'u}ster 1 ($\\psi_1$)",
    "C(estacion_climatica, Treatment('invierno'))[T.primavera]:C(region_geografica, Treatment('2'))[T.1]": "Primavera $\\times$ Cl{\\'u}ster 1 ($\\psi_2$)",
    "C(estacion_climatica, Treatment('invierno'))[T.verano]:C(region_geografica, Treatment('2'))[T.1]": "Verano $\\times$ Cl{\\'u}ster 1 ($\\psi_3$)",
    "tiempo": "Tiempo ($\\beta_1$)",
    "tiempo:C(estacion_climatica, Treatment('invierno'))[T.otono]": "Tiempo $\\times$ Oto{\\~n}o ($\\theta_1$)",
    "tiempo:C(estacion_climatica, Treatment('invierno'))[T.primavera]": "Tiempo $\\times$ Primavera ($\\theta_2$)",
    "tiempo:C(estacion_climatica, Treatment('invierno'))[T.verano]": "Tiempo $\\times$ Verano ($\\theta_3$)",
    "tiempo:C(region_geografica, Treatment('2'))[T.1]": "Tiempo $\\times$ Cl{\\'u}ster 1 ($\\phi_1$)",
    "tiempo:C(estacion_climatica, Treatment('invierno'))[T.otono]:C(region_geografica, Treatment('2'))[T.1]": "Tiempo $\\times$ Oto{\\~n}o $\\times$ Cl{\\'u}ster 1 ($\\omega_1$)",
    "tiempo:C(estacion_climatica, Treatment('invierno'))[T.primavera]:C(region_geografica, Treatment('2'))[T.1]": "Tiempo $\\times$ Primavera $\\times$ Cl{\\'u}ster 1 ($\\omega_2$)",
    "tiempo:C(estacion_climatica, Treatment('invierno'))[T.verano]:C(region_geografica, Treatment('2'))[T.1]": "Tiempo $\\times$ Verano $\\times$ Cl{\\'u}ster 1 ($\\omega_3$)"
}

def get_stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.1: return '*'
    return ''

# A. Imprimir Tabla de Parámetros en LaTeX
print("\n% ========================================================")
print("% CUADRO DE REGRESORES Y COEFICIENTES EN LATEX (DRISCOLL-KRAAY)")
print("% ========================================================")
print("\\begin{table}[ht!]")
print("\\centering")
print("\\caption{Resultados del modelo de panel con efectos aleatorios para la variable dependiente SPEI-6. Errores estándar de Driscoll y Kraay (1998) robustos a la correlación espacial.}")
print("\\label{tab:resultados_re}")
print("\\resizebox{\\textwidth}{!}{%")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("\\textbf{Regresor} & \\textbf{Coeficiente} & \\textbf{Error Est.} & \\textbf{Estadístico t} & \\textbf{p-valor} \\\\ \\hline")

for col in coef.index:
    name_clean = translation_map.get(col, col.replace('_', '\\_'))
    stars = get_stars(pvalues[col])
    print(f"{name_clean} & {coef[col]:.4f}{stars} & {std_err[col]:.4f} & {tstats[col]:.2f} & {pvalues[col]:.4f} \\\\")

print("\\hline")
print("\\multicolumn{5}{l}{\\small Nota: *** $p < 0.01$, ** $p < 0.05$, * $p < 0.1$. Categorías de referencia: Invierno y Clúster 2.} \\\\")
print("\\end{tabular}%")
print("}")
print("\\end{table}\n")

# B. Imprimir Tabla de Estadísticas de Ajuste (.pval corregido)
print("% ========================================================")
print("% CUADRO DE ESTADÍSTICAS GENERALES EN LATEX")
print("% ========================================================")
print("\\begin{table}[ht!]")
print("\\centering")
print("\\caption{Estadísticas de ajuste del modelo de efectos aleatorios estimado con errores de Driscoll-Kraay.}")
print("\\label{tab:estadisticas_modelo}")
print(r"\begin{tabular}{lc}")
print("\\hline")
print(r"\textbf{Métrica de Ajuste} & \textbf{Valor del Estimador} \\ \hline")
print(f"Número de Observaciones ($N \\times T$) & {re_model_dk.nobs} \\\\")
print(f"R-cuadrado global ($R^2$ overall) & {re_model_dk.rsquared:.4f} \\\\")
print(f"R-cuadrado dentro ($R^2$ within) & {re_model_dk.rsquared_within:.4f} \\\\")
print(f"R-cuadrado entre ($R^2$ between) & {re_model_dk.rsquared_between:.4f} \\\\")
print(f"Estadístico Wald ($\\chi^2$) & {re_model_dk.f_statistic.stat:.4f} \\\\")
print(f"p-valor de la regresión & {re_model_dk.f_statistic.pval:.4f} \\\\")
print("\\hline")
print("\\end{tabular}")
print("\\end{table}\n")


TEST DE DEPENDENCIA DE SECCIÓN CRUZADA DE PESARAN (2004)
Estadístico CD de Pesaran: 70.5878
p-valor resultante:        0.0000e+00
--------------------------------------------------------
Decisión: Se rechaza H0 (Independencia de Sección Cruzada).
Existe una fuerte correlación espacial/CSD entre estaciones.
Metodología: Se requiere el estimador robusto de Driscoll-Kraay.



/tmp/ipykernel_13245/3352580490.py:87: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

C(region_geografica, Treatment('2'))[T.1]

  fe_model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True).fit()


                        RandomEffects Estimation Summary                        
Dep. Variable:                 spei_6   R-squared:                        0.0356
Estimator:              RandomEffects   R-squared (Between):              0.1903
No. Observations:                5094   R-squared (Within):               0.0356
Date:                Wed, Jun 10 2026   R-squared (Overall):              0.0356
Time:                        09:01:09   Log-likelihood                   -7135.8
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      12.489
Entities:                          18   P-value                           0.0000
Avg Obs:                       283.00   Distribution:                 F(15,5078)
Min Obs:                       283.00                                           
Max Obs:                       283.00   F-statistic (robust):             1.9807
                            